# CardioIA - Fase 2 - Parte 2
## Classificador de risco a partir de frases de sintomas

Aqui eu treino um modelo que lê uma frase de paciente e responde **alto risco** ou
**baixo risco**, simulando a triagem que decide quem é atendido primeiro.

Caminho: TF-IDF para transformar texto em números, depois um classificador do
scikit-learn. Avalio com acurácia, matriz de confusão e validação cruzada.

**Aviso:** exercício acadêmico com base sintética. Não é ferramenta de triagem.

### 1. Carregando a base rotulada

`dados/frases_risco.csv` tem 80 frases escritas pelo grupo: 40 de alto risco e 40 de
baixo risco. Além de `frase` e `situacao`, existe uma coluna `grupo_semantico` — a
próxima seção explica para que ela serve.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__}")
assert sklearn.__version__ == "1.8.0", (
    "Escolha o kernel CardioIA Fase 2 (.venv), com scikit-learn 1.8.0. "
    "Outras versões podem alterar as divisões por grupo e os resultados."
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

# Reaproveito a normalização da Parte 1: mesma limpeza de acento e maiúscula.
from extracao_sintomas import normalizar

pd.set_option("display.max_colwidth", 80)

df = pd.read_csv(RAIZ / "dados" / "frases_risco.csv")
print(f"{len(df)} frases | {df.grupo_semantico.nunique()} grupos semânticos")
df.situacao.value_counts()

Python 3.13.7 | scikit-learn 1.8.0


80 frases | 40 grupos semânticos


situacao
alto risco     40
baixo risco    40
Name: count, dtype: int64

In [2]:
df.head(6)

,frase,situacao,grupo_semantico
0,Estou com uma dor forte no peito que desce pelo braço esquerdo e não passa d...,alto risco,dor_toracica_irradiada
1,"A dor no meu peito espalhou para o queixo e para o ombro esquerdo, começou f...",alto risco,dor_toracica_irradiada
2,dor no peito irradiando pro braco esquerdo com formigamento na mao,alto risco,dor_toracica_irradiada
3,"Sinto um aperto muito forte no peito mesmo parado no sofá, sem ter feito esf...",alto risco,dor_toracica_em_repouso
4,A dor no peito apareceu enquanto eu dormia e continua mesmo depois de eu lev...,alto risco,dor_toracica_em_repouso
5,De repente minha fala embolou e o lado direito do rosto caiu.,alto risco,sinais_neurologicos_subitos


In [3]:
# Tamanho dos grupos: de 1 a 3 frases cada.
df.grupo_semantico.value_counts().head(8)

grupo_semantico
dor_toracica_irradiada             3
sinais_neurologicos_subitos        3
dispneia_subita                    3
dor_musculoesqueletica_toracica    3
refluxo_azia                       3
sono_ruim                          3
sincope                            2
dor_toracica_em_repouso            2
Name: count, dtype: int64

### 2. Por que dividir por grupo, e não por frase

Variações da mesma ideia podem compartilhar vocabulário e estrutura. Se algumas
ficarem no treino e outras no teste, a avaliação pode ficar mais otimista.

A coluna `grupo_semantico` reúne essas variações. `StratifiedGroupKFold` mantém cada
grupo em um único lado da divisão e busca preservar a proporção das classes.
A proporção exata depende da composição dos grupos; conferimos a distribuição abaixo.

In [4]:
X, y, grupos = df["frase"], df["situacao"], df["grupo_semantico"]

separador = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
idx_treino, idx_teste = next(separador.split(X, y, groups=grupos))

X_treino, X_teste = X.iloc[idx_treino], X.iloc[idx_teste]
y_treino, y_teste = y.iloc[idx_treino], y.iloc[idx_teste]

print(f"treino: {len(X_treino)} frases | teste: {len(X_teste)} frases")
print("teste por classe:", y_teste.value_counts().to_dict())

treino: 60 frases | teste: 20 frases
teste por classe: {'alto risco': 10, 'baixo risco': 10}


In [5]:
# Conferência obrigatória: nenhum grupo pode aparecer nos dois lados.
vazando = set(grupos.iloc[idx_treino]) & set(grupos.iloc[idx_teste])
print("grupos presentes em treino E teste:", len(vazando))
assert len(vazando) == 0, "Vazamento entre treino e teste!"

grupos presentes em treino E teste: 0


### 3. TF-IDF: transformando frase em número

O TF-IDF representa cada frase por uma linha e cada termo por uma coluna. Combina a
frequência do termo na frase com um peso inverso à sua frequência entre documentos
de treinamento. Usamos frequência sublinear e normalização dos vetores.

- `preprocessor=normalizar` unifica acentuação e maiúsculas.
- `ngram_range=(1, 2)` inclui palavras e pares de palavras, como `em repouso`.

Bigramas representam combinações locais; não garantem compreensão do contexto.
O vocabulário é aprendido apenas no treinamento.

In [6]:
vetor = TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), sublinear_tf=True)
matriz = vetor.fit_transform(X_treino)
print(f"matriz: {matriz.shape[0]} frases x {matriz.shape[1]} termos")

matriz: 60 frases x 858 termos


In [7]:
# Como fica uma frase depois do TF-IDF: só os termos com peso maior que zero.
exemplo = X_treino.iloc[0]
linha = pd.Series(matriz[0].toarray()[0], index=vetor.get_feature_names_out())
print(exemplo, "\n")
linha[linha > 0].sort_values(ascending=False).head(10)

De repente minha fala embolou e o lado direito do rosto caiu. 



caiu             0.249183
do rosto         0.249183
direito do       0.249183
rosto caiu       0.249183
repente minha    0.249183
embolou lado     0.249183
embolou          0.249183
fala embolou     0.249183
rosto            0.249183
minha fala       0.249183
dtype: float64

### 4. Modelo 1: Regressão Logística

Usamos regressão logística como modelo principal por trabalhar com a matriz esparsa
do TF-IDF e permitir a inspeção dos coeficientes de cada termo.

`class_weight="balanced"` calcula pesos inversos à frequência das classes no treino.
Nesta divisão, as classes estão equilibradas e recebem pesos iguais. A configuração
não garante preferência por alto risco nem elimina falsos negativos.

In [8]:
def montar_pipeline(classificador):
    """TF-IDF e classificador no mesmo objeto.

    Usar Pipeline não é enfeite: garante que o TF-IDF seja ajustado SÓ com os dados de
    treino dentro de cada divisão da validação cruzada. Se eu ajustasse o vetorizador na
    base inteira antes de dividir, o vocabulário do teste vazaria para o treino.
    """
    return Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", classificador),
    ])

modelo_lr = montar_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
modelo_lr.fit(X_treino, y_treino)
pred_lr = modelo_lr.predict(X_teste)

print(f"Acurácia no teste: {accuracy_score(y_teste, pred_lr):.1%}")

Acurácia no teste: 90.0%


### 5. Matriz de confusão

A matriz separa os tipos de erro, usando os rótulos sintéticos da equipe como referência:

- **falso negativo:** frase rotulada como alto risco prevista como baixo risco;
- **falso positivo:** frase rotulada como baixo risco prevista como alto risco.

São erros de classificação de frases. A avaliação não mede desfechos de pacientes.
A matriz é mostrada como tabela para facilitar a leitura dos valores.

In [9]:
ordem = ["alto risco", "baixo risco"]
matriz_conf = pd.DataFrame(
    confusion_matrix(y_teste, pred_lr, labels=ordem),
    index=pd.Index(ordem, name="rotulo"),
    columns=pd.Index(ordem, name="previsao"),
)
matriz_conf

previsao,alto risco,baixo risco
rotulo,,
alto risco,10,0
baixo risco,2,8


In [10]:
print(classification_report(y_teste, pred_lr, digits=3))

              precision    recall  f1-score   support

  alto risco      0.833     1.000     0.909        10
 baixo risco      1.000     0.800     0.889        10

    accuracy                          0.900        20
   macro avg      0.917     0.900     0.899        20
weighted avg      0.917     0.900     0.899        20


In [11]:
# Traduzindo a matriz para os termos da triagem.
fn = matriz_conf.loc["alto risco", "baixo risco"] if "baixo risco" in matriz_conf.columns else 0
fp = matriz_conf.loc["baixo risco", "alto risco"] if "alto risco" in matriz_conf.columns else 0
print(f"Falsos negativos (grave tratado como leve) : {fn}")
print(f"Falsos positivos (leve tratado como grave) : {fp}")

Falsos negativos (grave tratado como leve) : 0
Falsos positivos (leve tratado como grave) : 2


### 6. Modelo 2: Árvore de Decisão, para comparar

O enunciado cita Decision Tree como alternativa. Treino a mesma coisa com árvore para
ver se a escolha do modelo muda o resultado.

In [12]:
modelo_dt = montar_pipeline(
    DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
)
modelo_dt.fit(X_treino, y_treino)
pred_dt = modelo_dt.predict(X_teste)

print(f"Árvore de Decisão - acurácia: {accuracy_score(y_teste, pred_dt):.1%}")
pd.crosstab(y_teste, pred_dt, rownames=["real"], colnames=["previsto"])

Árvore de Decisão - acurácia: 70.0%


previsto,alto risco,baixo risco
real,,
alto risco,8,2
baixo risco,4,6


### 7. Comparando com uma regra por palavras-chave

A regra classifica como alto risco quando encontra `peito`, `coracao`, `falta de ar`
ou `braco`; caso contrário, retorna baixo risco. Ela não aprende com os dados.

Comparar os modelos com essa referência ajuda a avaliar o benefício do treinamento
nesta base. Não valida o uso clínico de nenhum dos métodos.

In [13]:
palavras_chave = ["peito", "coracao", "falta de ar", "braco"]
pred_baseline = [
    "alto risco" if any(p in normalizar(f) for p in palavras_chave) else "baixo risco"
    for f in X_teste
]

print(f"Baseline por palavra-chave : {accuracy_score(y_teste, pred_baseline):.1%}")
print(f"Regressão Logística        : {accuracy_score(y_teste, pred_lr):.1%}")
print(f"Árvore de Decisão          : {accuracy_score(y_teste, pred_dt):.1%}")

Baseline por palavra-chave : 60.0%


Regressão Logística        : 90.0%
Árvore de Decisão          : 70.0%


### 8. Validação cruzada por grupo

O teste isolado usa apenas 20 frases. Seu resultado depende da divisão escolhida.
A validação cruzada avalia cinco divisões e informa a média e o desvio-padrão das
acurácias. O desvio-padrão não é um intervalo de confiança.

Essa avaliação amplia a cobertura dentro da base sintética. Continua sem avaliação
externa e não mede generalização para relatos de pacientes reais.

In [14]:
cv_grupo = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

cv_lr = cross_val_score(montar_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    X, y, groups=grupos, cv=cv_grupo)
cv_dt = cross_val_score(montar_pipeline(
    DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)),
    X, y, groups=grupos, cv=cv_grupo)

print(f"Regressão Logística : {cv_lr.mean():.1%} +/- {cv_lr.std():.1%}   folds: {np.round(cv_lr, 3)}")
print(f"Árvore de Decisão   : {cv_dt.mean():.1%} +/- {cv_dt.std():.1%}   folds: {np.round(cv_dt, 3)}")

Regressão Logística : 77.5% +/- 5.0%   folds: [0.812 0.688 0.812 0.75  0.812]


Árvore de Decisão   : 63.7% +/- 8.3%   folds: [0.688 0.625 0.75  0.625 0.5  ]


### 8.1 Matriz de confusão agregada e baseline nas mesmas divisões

A matriz da seção 5 vem de uma única divisão. Aqui reúno as previsões das cinco divisões
da validação cruzada: cada uma das 80 frases é prevista por um modelo que **não viu o grupo
dela no treino**. Isso dá uma visão mais completa dos tipos de erro.

Também avalio o baseline por palavras-chave nas **mesmas cinco divisões**, para que a
comparação com a regressão logística seja equivalente (o baseline não aprende nada, então
a acurácia dele em cada divisão é só a taxa de acerto da regra naquelas frases).

In [15]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_fscore_support

ordem = ["alto risco", "baixo risco"]

pred_cv = cross_val_predict(
    montar_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    X, y, groups=grupos, cv=cv_grupo,
)

matriz_cv = pd.DataFrame(
    confusion_matrix(y, pred_cv, labels=ordem),
    index=pd.Index(ordem, name="rotulo"),
    columns=pd.Index(ordem, name="previsao"),
)
print(f"Acurácia agregada: {accuracy_score(y, pred_cv):.1%}\n")
print(matriz_cv, "\n")
print(classification_report(y, pred_cv, labels=ordem, digits=3))
print(f"Falsos negativos (grave tratado como leve): {matriz_cv.loc['alto risco', 'baixo risco']}")
print(f"Falsos positivos (leve tratado como grave): {matriz_cv.loc['baixo risco', 'alto risco']}")

Acurácia agregada: 77.5%

previsao     alto risco  baixo risco
rotulo                              
alto risco           30           10
baixo risco           8           32 

              precision    recall  f1-score   support

  alto risco      0.789     0.750     0.769        40
 baixo risco      0.762     0.800     0.780        40

    accuracy                          0.775        80
   macro avg      0.776     0.775     0.775        80
weighted avg      0.776     0.775     0.775        80

Falsos negativos (grave tratado como leve): 10
Falsos positivos (leve tratado como grave): 8


In [16]:
def baseline(frases):
    return ["alto risco" if any(p in normalizar(f) for p in palavras_chave) else "baixo risco"
            for f in frases]

cv_base = np.array([
    accuracy_score(y.iloc[te], baseline(X.iloc[te]))
    for _, te in cv_grupo.split(X, y, groups=grupos)
])

print(f"Baseline palavra-chave : {cv_base.mean():.1%} +/- {cv_base.std():.1%}   folds: {np.round(cv_base, 3)}")
print(f"Regressão Logística    : {cv_lr.mean():.1%} +/- {cv_lr.std():.1%}   folds: {np.round(cv_lr, 3)}")
print(f"Diferença média        : {(cv_lr.mean() - cv_base.mean()) * 100:+.1f} pontos percentuais")

Baseline palavra-chave : 58.8% +/- 23.6%   folds: [0.875 0.188 0.625 0.75  0.5  ]
Regressão Logística    : 77.5% +/- 5.0%   folds: [0.812 0.688 0.812 0.75  0.812]
Diferença média        : +18.8 pontos percentuais


### 9. Comparando divisão por grupo e divisão por frase

Abaixo rodo a mesma validação cruzada duas vezes:

- **por grupo** (`StratifiedGroupKFold`): variações da mesma ideia ficam do mesmo lado;
- **por frase** (`StratifiedKFold`): o jeito comum, que deixa frases quase iguais
  caírem em lados opostos.

Se a divisão por frase der um número mais otimista, isso é compatível com o risco de
vazamento entre variações parecidas. A diferença sozinha, porém, não mede apenas o
vazamento: as duas estratégias também montam conjuntos de teste diferentes.

In [17]:
cv_vazando = cross_val_score(montar_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

print(f"CV dividindo por GRUPO (principal) : {cv_lr.mean():.2%}")
print(f"CV dividindo por FRASE (comparação): {cv_vazando.mean():.2%}")
print(f"Diferença observada                : {(cv_vazando.mean() - cv_lr.mean()) * 100:+.2f} pontos percentuais")

CV dividindo por GRUPO (principal) : 77.50%
CV dividindo por FRASE (comparação): 83.75%
Diferença observada                : +6.25 pontos percentuais


### 10. Onde o modelo errou

Olhar as frases erradas ensina mais do que a acurácia.

In [18]:
erros = pd.DataFrame({
    "frase": X_teste,
    "real": y_teste,
    "previsto": pred_lr,
    "grupo": grupos.iloc[idx_teste],
})
erros = erros[erros.real != erros.previsto]

if len(erros) == 0:
    print("Nenhum erro nesta divisão.")
else:
    for _, l in erros.iterrows():
        print(f"\nreal: {l['real']:12} previsto: {l['previsto']:12} [{l['grupo']}]")
        print(f"  {l['frase']}")


real: baixo risco  previsto: alto risco   [dor_musculoesqueletica_toracica]
  Fiz supino na academia e o músculo do peito está doendo ao levantar o braço.

real: baixo risco  previsto: alto risco   [dispneia_pos_exercicio]
  Corri cinco quilômetros e fiquei com falta de ar, mas normalizou em poucos minutos parado.


**O teste isolado teve dois falsos positivos e nenhum falso negativo.**

As duas frases rotuladas como baixo risco e previstas como alto risco incluem
`dor no peito` após academia e `falta de ar` após corrida. Essa observação descreve
apenas esta divisão de 20 frases.

Na matriz agregada da seção 8.1, há dez falsos negativos e oito falsos positivos.
Portanto, o zero observado no teste isolado não se mantém na avaliação completa.

### 11. Quais termos o modelo usa para decidir

Um dos motivos de eu ter escolhido Regressão Logística: dá para abrir e ver o peso de
cada termo. Atenção ao sinal: o scikit-learn ordena as classes em ordem alfabética,
`['alto risco', 'baixo risco']`, e o coeficiente se refere à segunda classe. Portanto,
**coeficiente positivo empurra para baixo risco e negativo empurra para alto risco.**
A célula abaixo imprime a ordem das classes para confirmar.

Os coeficientes mostram a direção de cada termo, mas a previsão combina todos os termos
da frase; nenhum coeficiente decide sozinho.

In [19]:
tfidf_ajustado = modelo_lr.named_steps["tfidf"]
coefs = pd.Series(
    modelo_lr.named_steps["clf"].coef_[0],
    index=tfidf_ajustado.get_feature_names_out(),
)
# A classe positiva é a segunda em ordem alfabética: 'baixo risco'.
print("classes na ordem do modelo:", modelo_lr.named_steps["clf"].classes_)

classes na ordem do modelo: ['alto risco' 'baixo risco']


In [20]:
print("Termos que mais puxam para BAIXO RISCO:")
print(coefs.sort_values(ascending=False).head(12).round(3).to_string())
print("\nTermos que mais puxam para ALTO RISCO:")
print(coefs.sort_values().head(12).round(3).to_string())

Termos que mais puxam para BAIXO RISCO:
da           0.418
quando       0.412
depois       0.408
quando eu    0.395
um           0.367
tempo        0.307
leve         0.281
um pouco     0.275
pouco        0.249
dolorido     0.246
doi          0.240
passou       0.234

Termos que mais puxam para ALTO RISCO:
nao           -0.511
nao consigo   -0.337
consigo       -0.337
ar            -0.335
para          -0.327
no peito      -0.311
junto         -0.264
no            -0.253
ha            -0.230
falta         -0.228
respirar      -0.225
agora         -0.224


**Os coeficientes sugerem sensibilidade ao estilo de escrita.**

Termos como `quando`, `depois`, `da` e `um` favorecem baixo risco no modelo treinado.
Eles aparecem em construções condicionais usadas pela equipe. Isso é um indício
de dependência da redação; não quantifica quanto da acurácia decorre desse padrão.

`nao` contribui para alto risco. Esse coeficiente isolado não determina a previsão
de `não sinto dor no peito`: a decisão combina todos os atributos da frase.
Os bigramas permitem capturar algumas combinações, sem garantir tratamento de negação.

Para investigar essas limitações, seria necessário coletar frases de diferentes
autores e comparar previsões para paráfrases e negações com rótulos revisados.

### 12. Testando com frases novas

Frases que não estão na base, incluindo casos de fronteira escritos de propósito.

In [21]:
frases_novas = [
    "Estou com dor no peito há quarenta minutos e o braço esquerdo está dormente.",
    "Meu peito doeu um pouco quando eu levantei o sofá ontem.",
    "nao consigo respirar e meus labios estao roxos",
    "Fiquei com dor de cabeça leve depois de um dia longo de trabalho.",
    "Senti o coração acelerar depois do café da manhã, passou logo.",
    "Acordei sufocando de madrugada e precisei sentar para conseguir respirar.",
]

probs = modelo_lr.predict_proba(frases_novas)
classes = modelo_lr.named_steps["clf"].classes_
i_alto = list(classes).index("alto risco")

for f, p in zip(frases_novas, probs):
    print(f"{p[i_alto]:5.1%} alto risco  |  {f}")

59.4% alto risco  |  Estou com dor no peito há quarenta minutos e o braço esquerdo está dormente.
35.0% alto risco  |  Meu peito doeu um pouco quando eu levantei o sofá ontem.
64.3% alto risco  |  nao consigo respirar e meus labios estao roxos
44.4% alto risco  |  Fiquei com dor de cabeça leve depois de um dia longo de trabalho.
37.4% alto risco  |  Senti o coração acelerar depois do café da manhã, passou logo.
57.3% alto risco  |  Acordei sufocando de madrugada e precisei sentar para conseguir respirar.


**As estimativas de alto risco ficaram entre aproximadamente 35% e 64,3%.**

Esses valores são saídas de `predict_proba` do modelo ajustado às 60 frases do treino
isolado. Não representam risco clínico e a calibração não foi avaliada.
Não existe probabilidade esperada próxima de 100% definida por este experimento.

O exemplo `nao consigo respirar e meus labios estao roxos` recebe cerca de 64,3%.
Um corte hipotético de 70% não o classificaria como alto risco. Isso ilustra como
um limiar altera a decisão; não fundamenta a escolha de limiares para atendimento.

### 13. Salvando os resultados

In [22]:
resultados = pd.DataFrame([
    {"modelo": "Regressão Logística", "acuracia_split_unico": accuracy_score(y_teste, pred_lr),
     "cv_media": cv_lr.mean(), "cv_desvio": cv_lr.std()},
    {"modelo": "Árvore de Decisão", "acuracia_split_unico": accuracy_score(y_teste, pred_dt),
     "cv_media": cv_dt.mean(), "cv_desvio": cv_dt.std()},
    {"modelo": "Baseline palavra-chave", "acuracia_split_unico": accuracy_score(y_teste, pred_baseline),
     "cv_media": cv_base.mean(), "cv_desvio": cv_base.std()},
]).round(4)
resultados.to_csv(RAIZ / "dados" / "resultado_classificador.csv", index=False, encoding="utf-8")

# Métricas por classe e matriz agregada da validação cruzada (seção 8.1).
prec, rec, f1, sup = precision_recall_fscore_support(y, pred_cv, labels=ordem)
metricas_cv = pd.DataFrame({
    "classe": ordem, "precisao": prec, "sensibilidade": rec, "f1": f1, "suporte": sup,
    "previsto_alto_risco": matriz_cv["alto risco"].values,
    "previsto_baixo_risco": matriz_cv["baixo risco"].values,
}).round(4)
metricas_cv.to_csv(RAIZ / "dados" / "metricas_cv_agregada.csv", index=False, encoding="utf-8")

display(resultados)
metricas_cv

,modelo,acuracia_split_unico,cv_media,cv_desvio
0,Regressão Logística,0.9,0.7750,0.0500
1,Árvore de Decisão,0.7,0.6375,0.0829
2,Baseline palavra-chave,0.6,0.5875,0.2358


,classe,precisao,sensibilidade,f1,suporte,previsto_alto_risco,previsto_baixo_risco
0,alto risco,0.7895,0.75,0.7692,40,30,10
1,baixo risco,0.7619,0.80,0.7805,40,8,32


### 14. Limitações, vieses e governança

1. **Base pequena e sintética:** 80 frases da equipe, sem pacientes reais nem
   avaliação externa. A acurácia mede concordância com os rótulos da base.
2. **Rótulos sem revisão clínica:** o gabarito foi definido pelos estudantes e
   sua qualidade limita a interpretação dos resultados.
3. **Risco de sobreajuste:** 858 termos para 60 frases no treino isolado. A diferença
   entre 90% no teste isolado e 77,5% na validação cruzada não comprova sobreajuste
   sozinha, pois as divisões também mudam as amostras de avaliação.
4. **Sensibilidade à redação:** os coeficientes indicam uso de termos de estilo.
   Não medimos o impacto desse padrão na acurácia nem diferenças entre populações.
5. **Negação e contexto limitados:** TF-IDF e bigramas não oferecem interpretação
   geral de negação, tempo ou sujeito. A decisão combina vários atributos.
6. **Calibração não avaliada:** `predict_proba` não fundamenta limiares clínicos.
7. **Sem dados de subgrupo:** não podemos medir diferenças por idade, região,
   escolaridade ou outros recortes populacionais.

O notebook demonstra extração de atributos, treinamento e avaliação de classificação
de texto. Não foi validado para diagnóstico ou triagem de pacientes.